# SafeDecoding — chạy thử trên Google Colab (GPU free)

Chạy lại method **SafeDecoding** (ACL 2024) trên sample nhỏ để **chứng minh method chạy + lấy số DSR**.

**Cách chạy:** `Runtime` → `Change runtime type` → **T4 GPU** → Save. Rồi `Runtime` → **Run all**.

_Không cần Restart runtime — mình cài vào 1 môi trường Python 3.10 RIÊNG (uv) nên không đụng Python của Colab._
_Lần đầu sẽ tải Vicuna-7B (~13GB) → đợi vài phút._


In [ ]:
# Cell 1: kiểm tra GPU (phải thấy 'Tesla T4'). Lỗi -> chưa bật GPU runtime.
!nvidia-smi


In [ ]:
# Cell 2: clone repo + tạo venv Python 3.10 (uv) + cài đúng version đã test
# (Colab base là Python 3.12, stack cũ của paper cần 3.10 -> dựng venv riêng)
!git clone -q https://github.com/PTAxHVA/safedecoding-reproduce.git /content/repo
!pip install -q uv
!uv venv --python 3.10 /content/sdvenv
%cd /content/repo
!uv pip install --python /content/sdvenv/bin/python -q torch==2.1.2
!uv pip install --python /content/sdvenv/bin/python -q -r requirements.txt
!/content/sdvenv/bin/python -c 'import torch,transformers; print("torch",torch.__version__,"| cuda",torch.cuda.is_available(),"| tf",transformers.__version__)'


In [ ]:
# Cell 3: CHẠY SafeDecoding (CÓ phòng thủ) trên 20 prompt sample
# Lần đầu tải Vicuna-7B (~13GB). Cuối log có dòng  ASR: xx.xx%  ->  DSR = 100 - ASR.
%cd /content/repo/exp
!/content/sdvenv/bin/python defense.py --model_name vicuna --attacker custom --defender SafeDecoding --disable_GPT_judge --max_new_tokens 128


In [ ]:
# Cell 4: (đối chứng) TẮT phòng thủ
!/content/sdvenv/bin/python defense.py --model_name vicuna --attacker custom --defense_off --disable_GPT_judge --max_new_tokens 128


## Đọc kết quả
- Dòng `ASR: xx.xx%` → **DSR = 100 − ASR**. File chi tiết ở `/content/repo/exp/exp_outputs/...`.

> ⚠️ **20 sample là prompt harmful THÔ (AdvBench)** → Vicuna vốn đã từ chối dù không phòng thủ, nên ASR ≈ 0 ở cả 2 → chưa thấy chênh lệch.
> Muốn thấy SafeDecoding *có tác dụng*, đổi `--attacker custom` → **`--attacker GCG`** (prompt đã bị jailbreak): no-defense ASR cao, SafeDecoding kéo xuống thấp. Hoặc cắm **data của anh** vào `datasets/custom_prompts.json`.

### Nếu lỗi
- **OOM**: giảm `--max_new_tokens 64`.
- Lỗi khác: copy log gửi lại.

### (tuỳ chọn) Harmful Score bằng Gemini
```python
import os; os.environ['GEMINI_API_KEY']='<key>'
!cd /content/repo && /content/sdvenv/bin/python gemini_judge.py --input exp/exp_outputs/<thư-mục>/<file>.json
```
